# Code Torique de Kitaev — Simulation numérique

**CY Tech · GM DATA · Juin 2026**  
*Superviseur : Garrigue*

Ce notebook simule la dynamique d'erreurs du code torique via un **canal de Pauli stochastique classique**.  
On travaille directement sur le **syndrome** (grille des valeurs propres $B_p \in \{-1,+1\}$), ce qui est exactement ce que voit un décodeur en pratique.

### Structure
1. Modèle et fonctions de base  
2. Simulation et visualisation — régime sous-critique  
3. Transition de phase : nombre d'anyons vs $p$  
4. Comparaison de régimes ($p$ petit / critique / sur-critique)


## 1. Modèle et fonctions de base

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.animation import FuncAnimation
from tqdm import tqdm

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.facecolor': '#0d0d0d',
    'figure.facecolor': '#1a1a1a',
    'text.color': 'white',
    'axes.labelcolor': 'white',
    'xtick.color': 'white',
    'ytick.color': 'white',
    'axes.titlecolor': 'white',
    'axes.edgecolor': '#444',
    'grid.color': '#333',
})


### Structure du réseau torique $L \times L$

Les qubits sont sur les **arêtes** du réseau. On distingue :
- arêtes **horizontales** $e = (i,j)_h$ — indice $i \cdot L + j$  
- arêtes **verticales** $e = (i,j)_v$ — indice $L^2 + i \cdot L + j$

Chaque arête est adjacente à exactement **2 plaquettes** (conditions périodiques = tore).


In [ ]:
def edge_h(i0, j0, L):
    """Arête horizontale en (i0, j0) mod L."""
    return (i0 % L) * L + (j0 % L), 'h'

def edge_v(i0, j0, L):
    """Arête verticale en (i0, j0) mod L."""
    return edge_h(i0, j0, L)[0] + L**2, 'v'

def adjacent_plaquettes(e, L):
    """Renvoie les 2 plaquettes adjacentes à l'arête e (indices mod L)."""
    i, j = e[0] // L, e[0] % L
    if e[1] == 'h':
        return [(i - 1) % L, j], [i % L, j]
    else:
        return [i % L, (j - 1) % L], [i % L, j % L]

# Test de cohérence : chaque arête doit avoir exactement 2 plaquettes adjacentes
L_test = 5
e = edge_h(2, 3, L_test)
p1, p2 = adjacent_plaquettes(e, L_test)
print(f"Arête {e} → plaquettes {p1} et {p2}  ✓")


### Canal de Pauli stochastique (Définition 8.1 du rapport)

À chaque pas de temps, chaque arête $e$ subit une erreur $X_e$ avec probabilité $p$, indépendamment.  
Une erreur sur $e$ **flip** les deux plaquettes adjacentes dans le syndrome :  
$$B_p \mapsto -B_p \quad \text{pour } p \text{ adjacent à } e.$$

**État** : `syndrome[i,j]` $\in \{-1, +1\}$ — valeur propre de $B_{(i,j)}$.  
Une plaquette avec `syndrome = -1` **contient un anyon**.


In [ ]:
def step(syndrome, p, L):
    """Un pas de temps du canal de Pauli stochastique."""
    for i in range(L):
        for j in range(L):
            # Arête horizontale (i,j)
            if np.random.rand() < p:
                p1, p2 = adjacent_plaquettes(edge_h(i, j, L), L)
                syndrome[p1[0], p1[1]] *= -1
                syndrome[p2[0], p2[1]] *= -1
            # Arête verticale (i,j)
            if np.random.rand() < p:
                p1, p2 = adjacent_plaquettes(edge_v(i, j, L), L)
                syndrome[p1[0], p1[1]] *= -1
                syndrome[p2[0], p2[1]] *= -1
    return syndrome

def n_anyons(syndrome):
    """Nombre de plaquettes excitées (anyons). Toujours pair — Proposition 9.2."""
    return int(np.sum(syndrome == -1))

def simulate(L, p, n_steps, init_anyons=None):
    """
    Simule n_steps pas du canal de Pauli sur un réseau L×L.
    init_anyons : liste de (i,j) à initialiser à -1 (paires d'anyons).
    Retourne l'historique des syndromes.
    """
    syndrome = np.ones((L, L), dtype=np.int8)
    if init_anyons:
        for (i, j) in init_anyons:
            syndrome[i % L, j % L] = -1
    history = [syndrome.copy()]
    for _ in range(n_steps):
        syndrome = step(syndrome, p, L)
        history.append(syndrome.copy())
    return history


### Vérification : parité des anyons (Proposition 9.2)

Le nombre d'anyons est **toujours pair**, quelle que soit la suite d'erreurs.  
Raison : $\prod_p B_p = I$, donc le produit de toutes les valeurs propres vaut $+1$,  
ce qui impose un nombre pair de valeurs propres $-1$.


In [ ]:
# Vérification rapide sur un petit réseau
L_v, p_v, n_v = 10, 0.05, 200
hist_v = simulate(L_v, p_v, n_v, init_anyons=[(0,0),(5,5)])
counts = [n_anyons(s) for s in hist_v]
all_even = all(c % 2 == 0 for c in counts)
print(f"Nombre d'anyons toujours pair : {all_even}  ✓")
print(f"Min={min(counts)}, Max={max(counts)}, Moyenne={np.mean(counts):.1f}")


## 2. Simulation et animation — régime sous-critique

On simule avec $L = 30$, $p = 0.01$ (régime sous-critique, $p \ll p_{\text{seuil}} \approx 0.103$).  
On initialise deux paires d'anyons pour rendre la dynamique initiale visible.

**Panneau gauche** : heatmap du syndrome (jaune = anyon).  
**Panneau droit** : évolution du nombre total d'anyons.


In [ ]:
L = 30
p = 0.01
n_steps = 300

print(f"Simulation : L={L}, p={p}, {n_steps} pas...")
history = simulate(L, p, n_steps,
                   init_anyons=[(L//2, L//2), (L//2+2, L//2+2),
                                (2, 2), (5, 5)])
anyons_frames = [(1 - s) / 2 for s in history]
energies = [n_anyons(s) for s in history]
print(f"Fait. Anyons initiaux : {energies[0]}, final : {energies[-1]}")


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#1a1a1a')

# — Heatmap syndrome —
im = ax1.imshow(anyons_frames[0], vmin=0, vmax=1,
                cmap='inferno', interpolation='nearest',
                extent=[0, L, L, 0])
cbar = fig.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)
cbar.set_label("Anyon présent", color='white')
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')
ax1.set_title(f"Syndrome — réseau {L}×{L}", fontsize=12)
ax1.set_xlabel("j"); ax1.set_ylabel("i")
step_text = ax1.text(0.02, 0.97, '', transform=ax1.transAxes,
                     color='white', fontsize=9, va='top')

# — Courbe énergie —
ax2.set_facecolor('#0d0d0d')
ax2.set_xlim(0, n_steps + 1)
ax2.set_ylim(0, max(energies) * 1.15 + 2)
ax2.set_xlabel("Pas de temps")
ax2.set_ylabel("Nombre d'anyons")
ax2.set_title(f"Dynamique des anyons  (p = {p})", fontsize=12)
ax2.axhline(np.mean(energies[50:]), color='cyan', lw=1,
            linestyle='--', alpha=0.6, label=f'Moyenne = {np.mean(energies[50:]):.1f}')
ax2.legend(fontsize=8)
line, = ax2.plot([], [], color='#ff6b35', lw=1.5)

def update(frame):
    im.set_array(anyons_frames[frame])
    step_text.set_text(f'Pas {frame}  |  {energies[frame]} anyons')
    line.set_data(range(frame + 1), energies[:frame + 1])
    return im, line, step_text

ani = FuncAnimation(fig, update, frames=len(anyons_frames),
                    interval=40, blit=True)
plt.tight_layout()
plt.show()
# ani.save('toric_code.gif', writer='pillow', fps=25)  # décommenter pour sauvegarder


## 3. Transition de phase : densité d'anyons vs $p$

On mesure la **densité stationnaire d'anyons** $\langle n_{\text{anyons}} \rangle / L^2$  
en fonction de $p$, pour plusieurs tailles $L$.

- Pour $p < p_{\text{seuil}} \approx 0.103$ : densité faible, anyons isolés.  
- À $p_{\text{seuil}}$ : **transition de phase** — amas sans échelle caractéristique (percolation).  
- Pour $p > p_{\text{seuil}}$ : amas percolant → erreurs logiques inévitables.

*(Dennis et al., 2002 [2] dans le rapport)*


In [ ]:
# Paramètres du scan (réduits pour la rapidité — augmenter pour la précision)
p_values = np.linspace(0.01, 0.25, 25)
L_values = [10, 20, 30]
n_eq   = 100   # pas d'équilibration (ignorés)
n_meas = 200   # pas de mesure

results = {}  # {L: densités moyennes}

for Ls in L_values:
    print(f"L = {Ls}...", end=' ', flush=True)
    dens = []
    for pv in p_values:
        syn = np.ones((Ls, Ls), dtype=np.int8)
        # équilibration
        for _ in range(n_eq):
            syn = step(syn, pv, Ls)
        # mesure
        counts = []
        for _ in range(n_meas):
            syn = step(syn, pv, Ls)
            counts.append(n_anyons(syn) / Ls**2)
        dens.append(np.mean(counts))
    results[Ls] = dens
    print("✓")

print("Scan terminé.")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor('#1a1a1a')
ax.set_facecolor('#0d0d0d')

colors = ['#ff6b35', '#ffd166', '#06d6a0']
for (Ls, col) in zip(L_values, colors):
    ax.plot(p_values, results[Ls], 'o-', color=col,
            lw=2, ms=5, label=f'L = {Ls}')

ax.axvline(0.103, color='white', lw=1.5, linestyle='--', alpha=0.8)
ax.text(0.106, ax.get_ylim()[1]*0.95 if ax.get_ylim()[1] > 0 else 0.45,
        r'$p_{m seuil} pprox 0.103$', color='white', fontsize=10)

ax.set_xlabel("Probabilité d'erreur $p$", fontsize=12)
ax.set_ylabel(r"Densité d'anyons $\langle n angle / L^2$", fontsize=12)
ax.set_title("Transition de phase du code torique", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 4. Comparaison visuelle des trois régimes

Snapshots du syndrome après équilibration pour trois valeurs de $p$ :

| Régime | $p$ | Observation |
|---|---|---|
| Sous-critique | $0.01$ | Anyons rares, isolés |
| Critique | $0.103$ | Amas sans échelle caractéristique |
| Sur-critique | $0.20$ | Réseau quasi-saturé, erreurs logiques probables |


In [ ]:
L_snap = 40
p_snap = [0.01, 0.103, 0.20]
labels = ['Sous-critique  $p=0.01$',
          'Critique  $p=0.103$',
          'Sur-critique  $p=0.20$']
n_eq_snap = 500

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.patch.set_facecolor('#1a1a1a')
fig.suptitle(f"Syndrome après équilibration — réseau {L_snap}×{L_snap}",
             color='white', fontsize=13)

for ax, pv, lab in zip(axes, p_snap, labels):
    syn = np.ones((L_snap, L_snap), dtype=np.int8)
    for _ in range(n_eq_snap):
        syn = step(syn, pv, L_snap)
    snap = (1 - syn) / 2
    ax.imshow(snap, vmin=0, vmax=1, cmap='inferno',
              interpolation='nearest')
    ax.set_title(lab, fontsize=10, pad=6)
    ax.set_xticks([]); ax.set_yticks([])
    n = n_anyons(syn)
    ax.set_xlabel(f"{n} anyons  ({100*n/L_snap**2:.1f}%)", color='white')

plt.tight_layout()
plt.show()


## Bilan

| Ce que la simulation montre | Lien avec le rapport |
|---|---|
| Anyons toujours en nombre pair | Proposition 9.2 |
| Régime sous-critique : anyons rares et localisés | Section 12 — distance $d = L$ |
| Transition à $p \approx 0.103$ | Proposition 11.1, Dennis et al. [2] |
| Régime sur-critique : saturation → erreurs logiques | Théorème 12.1 |

**Ce que la simulation ne montre pas** : la diffusion au sens quantique rigoureux,  
ni la mesure quantitative de l'invariance d'échelle à la transition —  
ces affirmations nécessiteraient des mesures de percolation supplémentaires.
